In [1]:
import os
from summarization.prune import prune_graph_pipeline, PruneGraph
# from summarization.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import shap

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [4]:
explainer = shap.Explainer(model, tokenizer)

In [5]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [ ]:
shap_values = explainer(prompts)

In [ ]:
shap.plots.text(shap_values)

In [52]:
print(shap_values)

.values =
array([[[-4.22446732e-06],
        [ 6.54286429e+00],
        [ 1.68508452e+00],
        [ 1.20230146e+00],
        [ 6.04736818e+00],
        [ 6.77137808e-02],
        [ 4.40673760e+00],
        [ 4.12085123e-01],
        [ 4.40843961e-01]]])

.base_values =
array([[-21.73993724]])

.data =
(array(['', 'Cat', ' is', ' to', ' kitten', ' as', ' dog', ' is', ' to'],
      dtype=object),)


In [34]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [4]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="austin_plt",
    source_set = 'gemmascope-transcoder-16k',
    out_path="temp_graph_files/austin_plt.json",
    node_threshold = 0.95,
    edge_threshold = 0.98,
)

requesting graph for slug='austin_plt' prompt='Fact: The capital of the state containing Dallas is...'


downloading graph json from https://neuronpedia-attrib.s3.us-east-1.amazonaws.com/user-graphs/anonymous/austin_plt.json
saved austin_plt -> temp_graph_files/austin_plt.json (nodes=5387)


## ASG Pipeline

### Prune

In [26]:
from summarization.prune import prune_graph_pipeline

# prune_graph = load_prune_graph('subgraph/puppy_clt_shap_07_085.pt')
token_weights = [0,0,0,0,1/3,0,0,1/3,0,1/3,0]
# 1) Build prune_graph as usual
prune_graph = prune_graph_pipeline(
    json_path="temp_graph_files/austin_plt.json",
    token_weights = token_weights,
    logit_weights="target",
    node_influence_threshold=0.67,
    node_relevance_threshold=0.67,
    edge_influence_threshold=0.98,
    edge_relevance_threshold=0.98,
    keep_all_tokens_and_logits=False,
    filter_act_density=False,
)


In [27]:
print(prune_graph.edge_influence[prune_graph.pruned_adj != 0].max())
# print(prune_graph.edge_relevance)

tensor(0.2260)


In [30]:
num_nodes = prune_graph.num_nodes
num_edges = (prune_graph.pruned_adj != 0).sum().item()
num_edges_positive = (prune_graph.pruned_adj > 0).sum().item()
num_edges_negative = (prune_graph.pruned_adj < 0).sum().item()
print(f"Number of nodes: {num_nodes}")
print(f"Total nonzero edges: {num_edges}")
print(f"Edges with weight > 0: {num_edges_positive}")
print(f"Edges with weight < 0: {num_edges_negative}")

Number of nodes: 54
Total nonzero edges: 501
Edges with weight > 0: 397
Edges with weight < 0: 104


In [31]:
for node_id in prune_graph.kept_ids:
    print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_influence[prune_graph.kept_ids.index(node_id)].item(), prune_graph.node_relevance[prune_graph.kept_ids.index(node_id)].item())


16_25_9 Texas legal documents 0.01325109601020813 0.009252252988517284
16_12678_9 cities 0.0019933998119086027 0.007337966933846474
17_7178_10 government buildings 0.002735452726483345 0.00721630547195673
18_1026_10 country names 0.0019005164504051208 0.00769460154697299
18_1437_10 Legal documents from Texas 0.004528920166194439 0.013160908594727516
18_5495_10 locations 0.004346250090748072 0.00858826283365488
18_6101_10 capital cities 0.007259448990225792 0.011009098030626774
18_8959_10 government/state 0.007685741409659386 0.012188401073217392
19_7477_9 Dallas 0.0022130138240754604 0.02055974304676056
19_37_10 Places 0.0024963715113699436 0.00792014691978693
19_1445_10 Downtowns of cities 0.004945716354995966 0.01178178284317255
19_2695_10 cities 0.007433949038386345 0.012315378524363041
19_7477_10 Dallas 0.0033531615044921637 0.01701117679476738
20_15589_9 Texas 0.012131176888942719 0.02564992755651474
20_114_10 Oklahoma locations 0.001368592376820743 0.013856777921319008
20_5916_10

## Cluster

In [54]:
from summarization.prune import save_prune_graph, load_prune_graph
# save_prune_graph(prune_graph, "subgraph/austin_clt_clean_5_9_5.pt")
prune_graph = load_prune_graph("subgraph/austin_clt_clean_5_9_5.pt")

In [64]:
from summarization.cluster import cluster_graph, cluster_graph_with_labels
from summarization.auto_grouping import find_best_k
# 2) Auto-select k
best_k, sweep = find_best_k(
    prune_graph,
    max_layer_span=4,
    gamma=0.5,
    mediation_penalty=0,
    enforce_dag=False,              # optional DAG behavior
    use_flow_faithfulness=False,     # Stage 4 guided K search
    w_flow=0.40,                    # how much F(phi) affects total
)

print("best_k:", best_k)
print("best score:", sweep[best_k]["total"] if best_k in sweep else None)

# 3) Run final clustering with best_k
supernodes = cluster_graph_with_labels(
    prune_graph,
    target_k=best_k,
    max_layer_span=4,
    gamma=0.5,
    mediation_penalty=0.1,
    enforce_dag=True,
)

best_k: 4
best score: 0.8800154796942499


In [65]:
supernodes

[['cluster_3', '17_1822_10', '19_54790_10'],
 ['cluster_4', '17_98126_10', '18_3623_10'],
 ['cluster_5', '20_44686_9', '20_44686_10'],
 ['cluster_7', '22_11998_10', '22_74186_10', '22_81571_10'],
 ['cluster_9', '22_43081_10', '23_29602_10', '23_31673_10']]

In [57]:
for supernode in supernodes:
    print(supernode[0])
    for node_id in supernode[1:]:
        print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_scores[prune_graph.kept_ids.index(node_id)].item())

cluster_3
17_1822_10 place names 0.013156678527593613
19_54790_10 Places 0.010641026310622692
cluster_4
17_98126_10 Government buildings/institutions 0.01083023939281702
18_3623_10 capital cities 0.015388197265565395
cluster_5
20_44686_9 Texas locations/legal contexts 0.02212010696530342
20_44686_10 Texas locations/legal contexts 0.028163056820631027
cluster_7
22_11998_10 Texas and Dallas 0.023699676617980003
22_74186_10 general news snippets 0.011260032653808594
22_81571_10 Texas locations and organizations 0.012025214731693268
cluster_9
22_43081_10 technical contexts 0.011151234619319439
23_29602_10 City or region names 0.01013106293976307
23_31673_10 Court appeals at specific location 0.013401541858911514


### Visualize on the web

In [66]:
import json
from api import save_subgraph

status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug=prune_graph.metadata["slug"],                       # parent graph slug
    displayName="austin-baseline",     # subgraph name shown on Neuronpedia
    pinnedIds=prune_graph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.8,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

{'Content-Type': 'application/json', 'x-api-key': 'sk-np-Vi0OcQl8zNm9jC7nyGRYfwssvSakMyrPjV675uEWIuU0'}
status: 200
{'success': True, 'subgraphId': 'cmo5mvtbc000013to8x5c7pqy'}


# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality